In [1]:
import csv
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torchvision import transforms, models
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score
from torchvision import datasets
import os

In [2]:
import torch.nn as nn
from torchvision import models

class VGG11(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = models.vgg11(weights=models.VGG11_Weights.IMAGENET1K_V1)
        num_features = self.model.classifier[6].in_features
        self.model.classifier[6] = nn.Linear(num_features, 2)  

    def forward(self, x):
        return self.model(x)

In [3]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

train_dataset = datasets.ImageFolder("/kaggle/input/fakeface-train-data-v2", transform=transform)
val_dataset   = datasets.ImageFolder("/kaggle/input/fakeface-valid-data-v2", transform=transform)
test_dataset = datasets.ImageFolder("/kaggle/input/fakeface-test-data-v2", transform=transform)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=2)
test_loader   = DataLoader(test_dataset, batch_size=64, shuffle=False,num_workers=2)

In [4]:
import time
import torch
import torch.nn as nn
from torchvision import models
from tqdm import tqdm
from sklearn.metrics import roc_auc_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

standard_vgg11 = VGG11() 
standard_vgg11 = nn.DataParallel(standard_vgg11)
standard_vgg11 = standard_vgg11.to(device)

optimizer = torch.optim.Adam(standard_vgg11.parameters(), lr=5e-5)
criterion = nn.CrossEntropyLoss()

epoch_times = []
start_training = time.time()

for epoch in range(10):
    start_epoch = time.time()
    
    standard_vgg11.train()
    total_loss = 0
    total_batches = len(train_loader)
    
    for batch_idx, (x, y) in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1}", leave=False), start=1):
        x, y = x.to(device), y.to(device)
        loss = criterion(standard_vgg11(x), y)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        total_loss += loss.item()
    
    avg = total_loss / len(train_loader)

    standard_vgg11.eval()
    correct, total = 0, 0
    all_labels, all_probs = [], [] 
    
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            outputs = standard_vgg11(x)
            probs = torch.softmax(outputs, dim=1)[:, 1] 
            
            preds = torch.argmax(outputs, dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)

            all_labels.extend(y.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            
    val_acc = correct / total
    val_auc = roc_auc_score(all_labels, all_probs)

    epoch_time = time.time() - start_epoch
    epoch_times.append(epoch_time)
    
    print(f"[Standard VGG11] Epoch {epoch+1} | Train Loss: {avg:.4f} | Val Acc: {val_acc:.4f} | Val AUC: {val_auc:.4f} | Time: {epoch_time:.2f}s")

total_time = time.time() - start_training
avg_time = sum(epoch_times) / len(epoch_times)

print(f"\nTotal training time: {total_time:.2f}s")                     
print(f"Average time per epoch: {avg_time:.2f}s")

torch.save(standard_vgg11.state_dict(), "/kaggle/working/vgg11.pth")

Downloading: "https://download.pytorch.org/models/vgg11-8a719046.pth" to /root/.cache/torch/hub/checkpoints/vgg11-8a719046.pth
100%|██████████| 507M/507M [00:02<00:00, 179MB/s]


[Standard VGG11] Epoch 1 | Train Loss: 0.1989 | Val Acc: 0.9539 | Val AUC: 0.9906 | Time: 784.04s


[Standard VGG11] Epoch 2 | Train Loss: 0.0245 | Val Acc: 0.9572 | Val AUC: 0.9921 | Time: 744.44s


[Standard VGG11] Epoch 3 | Train Loss: 0.0143 | Val Acc: 0.9588 | Val AUC: 0.9931 | Time: 743.32s


[Standard VGG11] Epoch 4 | Train Loss: 0.0113 | Val Acc: 0.9613 | Val AUC: 0.9937 | Time: 743.21s


...

[Standard VGG11] Epoch 6 | Train Loss: 0.0084 | Val Acc: 0.9385 | Val AUC: 0.9928 | Time: 742.14s


[Standard VGG11] Epoch 7 | Train Loss: 0.0073 | Val Acc: 0.9641 | Val AUC: 0.9941 | Time: 742.24s


[Standard VGG11] Epoch 8 | Train Loss: 0.0062 | Val Acc: 0.9630 | Val AUC: 0.9933 | Time: 741.33s


[Standard VGG11] Epoch 9 | Train Loss: 0.0070 | Val Acc: 0.9508 | Val AUC: 0.9941 | Time: 741.46s


[Standard VGG11] Epoch 10 | Train Loss: 0.0042 | Val Acc: 0.9625 | Val AUC: 0.9955 | Time: 740.32s

Total training time: 7464.88s
Average time per epoch: 746.49s


In [5]:
from sklearn.metrics import classification_report

def evaluate_teacher(model, dataloader, device='cuda'):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for x, y in dataloader:
            x = x.to(device)
            y = y.cpu().numpy() 
            outputs = model(x)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()

            all_preds.extend(preds)
            all_labels.extend(y)

    print("\n=== Classification Report ===")
    print(classification_report(all_labels, all_preds, digits=4))

In [6]:
evaluate_teacher(standard_vgg11, test_loader)


=== Classification Report ===
              precision    recall  f1-score   support

           0     0.9476    0.9817    0.9643     20000
           1     0.9810    0.9457    0.9630     20000

    accuracy                         0.9637     40000
   macro avg     0.9643    0.9637    0.9637     40000
weighted avg     0.9643    0.9637    0.9637     40000

